# Importing

In [3]:
import pandas as pd
import numpy as np

In [4]:
data = pd.read_csv("../Data/processed_data.csv")
df = pd.DataFrame(data)

In [5]:
df.head()

,price,bedrooms,bathrooms,surface_area,latitude,longitude,furnished?,property_mortgaged?,city_Al Khoms,city_Benghazi,...,lister_type_Owner,facade_Eastern,facade_Northeast,facade_Northern,facade_Northwest,facade_Southeast,facade_Southern,facade_Southwest,facade_Unknown,facade_Western
0,13.487008,3.0,2.0,4.663439,32.847191,13.125021,False,False,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.493928,3.0,2.0,4.663439,32.778309,13.264442,False,False,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,12.577640,4.0,2.0,5.049856,32.763538,13.181585,False,False,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,12.345839,4.0,2.0,5.081404,32.781060,13.078820,False,False,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,13.017005,2.0,1.0,4.795791,32.829285,13.102837,False,False,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Preparing

In [6]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

### Initializing Random Seed Control

In [7]:
rs = 42

### Data split

In [8]:
X = df.drop("price", axis=1)
y = df["price"]
kf = KFold(n_splits=10, shuffle=True, random_state=rs)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=rs
)

# Modeling

In [10]:
import xgboost as xgb
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

### Base Model

In [11]:
class CityMeanRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, city_columns):
        self.city_columns = city_columns

    def fit(self, X_train, y_train):
        # Reconstruct city label from one-hot encoding
        city_labels = X_train[self.city_columns].idxmax(axis=1)

        # Compute mean price per city using TRAIN data only
        self.city_means_ = (
            pd.DataFrame({"city": city_labels, "price": y_train})
            .groupby("city")["price"]
            .mean()
        )

        # Global fallback mean
        self.global_mean_ = np.mean(y_train)

        return self

    def predict(self, X_train):
        city_labels = X_train[self.city_columns].idxmax(axis=1)

        return city_labels.map(self.city_means_).fillna(self.global_mean_).values

In [12]:
city_columns = [col for col in X_train.columns if col.startswith("city_")]

In [13]:
baseline_model = CityMeanRegressor(city_columns=city_columns)


### XGBoost

In [14]:
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",  # important for regression
    random_state=42,
    n_jobs=-1
)

# Validation

In [15]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

## Grid Searching

In [16]:
def cross_validate_model(model, X, y, cv):
    scores = cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    return -scores

### Base Model

In [17]:
baseline_scores = cross_validate_model(baseline_model, X_train, y_train, kf)

### Linear Regression

In [18]:
lr = LinearRegression()
lr_scores = cross_validate_model(lr, X_train, y_train, kf)

### Random Forest

In [19]:
rf = RandomForestRegressor(random_state=rs)
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [20]:
rf_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1

### Gradient Boosting

In [21]:
gb = GradientBoostingRegressor(random_state=42)
gb_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5]
}

gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=gb_param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [22]:
gb_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoost...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1], 'max_depth': [3, 5], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the compu

### XGBoost

In [23]:
param_grid = {
    "n_estimators": [200, 300, 500],
    "learning_rate": [0.03, 0.05, 0.07, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

In [24]:
grid_search.fit(X_train, y_train)

Fitting 10 folds for each of 288 candidates, totalling 2880 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [3, 5, ...], 'min_child_weight': [1, 3], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more mess

## Comparison

### Baseline

In [25]:
baseline_rmse_scores = -baseline_scores

print("RMSE per fold:", baseline_rmse_scores)
print("Mean RMSE:", baseline_rmse_scores.mean())
print("Std RMSE:", baseline_rmse_scores.std())

RMSE per fold: [-0.75348544 -0.75698008 -0.76143454 -0.80806802 -0.76117978 -0.78528375
 -0.7180143  -0.74667116 -0.675934   -0.79829879]
Mean RMSE: -0.7565349839596633
Std RMSE: 0.036550096747309795


### Linear Regression

In [26]:
print("Linear Regression RMSE:", -lr_scores.mean())

Linear Regression RMSE: -0.7108560106179024


### Random Forest

In [27]:
best_rf = rf_grid.best_estimator_

print("Best Parameters:", rf_grid.best_params_)
print("Best CV RMSE:", -rf_grid.best_score_)

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV RMSE: 0.6345357902979594


### Gradient Boosting

In [28]:
best_gb = gb_grid.best_estimator_

print("Best Parameters:", gb_grid.best_params_)
print("Best CV RMSE:", -gb_grid.best_score_)

Best Parameters: {'learning_rate': 0.01, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 200}
Best CV RMSE: 0.6451941288024762


### XGBoost

In [29]:
best_xgb = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)


Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8}
Best CV RMSE: 0.6356183259653438


# Feature reduction (NO significant improvment noticed)

In [30]:
from sklearn.inspection import permutation_importance

In [31]:
best_rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

## pruning

### Extract feature importance

In [32]:
def find_feature_importance(model, X_train, y_train):
    result = permutation_importance(model, X_train, y_train, n_repeats=10, random_state=42)
    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": result.importances_mean
    }).sort_values(by="importance", ascending=False)
    return importance_df

In [33]:
feature_importance_df = find_feature_importance(best_rf, X_train, y_train)
feature_importance_df

,feature,importance
3,latitude,0.491075
2,surface_area,0.383707
4,longitude,0.280399
1,bathrooms,0.208693
0,bedrooms,0.087592
15,subcategory_Villa,0.040172
16,lister_type_Agent,0.024065
18,lister_type_Owner,0.022063
19,facade_Eastern,0.021774
14,subcategory_House,0.018685


### Remove features with importance < 0.01

In [34]:
threshold = 0.01

important_features = feature_importance_df[
    feature_importance_df["importance"] >= threshold
]["feature"].tolist()

X_reduced = X_train[important_features]

### re-run cross-validation

In [35]:
rf_reduced_scores = cross_validate_model(best_rf, X_reduced, y_train, kf)
print("Reduced Feature RMSE:", -rf_reduced_scores.mean())
print("Best CV RMSE:", -rf_grid.best_score_)

Reduced Feature RMSE: -0.6357160046048366
Best CV RMSE: 0.6345357902979594


# Ensamble Model

In [36]:
from sklearn.ensemble import VotingRegressor

In [37]:
rf_gb_model = VotingRegressor(
    estimators=[
        ("rf", best_rf),
        ("gb", best_gb)
    ]
)

rf_gb_scores = cross_validate_model(rf_gb_model, X_train, y_train, kf)

print("Voting RMSE:", -rf_gb_scores.mean())


Voting RMSE: -0.6360732973377659


Comparing
• Mean ± std
• Spread of performance
• Stability across folds

In [38]:
rf_xgb_model = VotingRegressor(
    estimators=[
        ("rf", best_rf),
        ("xgb", best_xgb)
    ],
    weights=[2, 1]
)

rf_xgb_scores = cross_validate_model(rf_xgb_model, X_train, y_train, kf)

print("RMSE per fold:", rf_xgb_scores)
print("Mean RMSE:", rf_xgb_scores.mean())
print("Std RMSE:", rf_xgb_scores.std())


RMSE per fold: [0.59751818 0.63674891 0.69402958 0.63492408 0.63762631 0.61510641
 0.62468035 0.6782523  0.58551157 0.6368413 ]
Mean RMSE: 0.6341238993705166
Std RMSE: 0.031172377216551264


In [39]:
rf_scores = cross_validate_model(best_rf, X_train, y_train, kf)

print("RF RMSE per fold:", rf_scores)
print("RF Mean RMSE:", rf_scores.mean())
print("RF Std RMSE:", rf_scores.std())


RF RMSE per fold: [0.60014438 0.64150056 0.69596266 0.62893702 0.63728248 0.61559766
 0.62499122 0.67874072 0.5896869  0.6325143 ]
RF Mean RMSE: 0.6345357902979594
RF Std RMSE: 0.030761607169988776


# Testing

### Ensable Model

In [40]:
rf_xgb_model.fit(X_train, y_train)

test_predictions = rf_xgb_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("Test RMSE:", test_rmse)


Test RMSE: 0.6645405109211594


### Random Forest

In [41]:
best_rf.fit(X_train, y_train)

test_predictions = best_rf.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("Test RMSE:", test_rmse)

Test RMSE: 0.6670163170173896


# New Angle- Clusters (NO significant improvment noticed)

In [42]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

### Training Model

In [43]:
# Extract coordinates
coords_train = X_train[["latitude", "longitude"]]
coords_test = X_test[["latitude", "longitude"]]

# Scale
scaler = StandardScaler()
coords_train_scaled = scaler.fit_transform(coords_train)
coords_test_scaled = scaler.transform(coords_test)

In [44]:
# Choose number of clusters (start simple)
kmeans = KMeans(
    n_clusters=10,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42
)

In [45]:
# Fit only on training data
train_clusters = kmeans.fit_predict(coords_train_scaled)
test_clusters = kmeans.predict(coords_test_scaled)

#### Adding Cluster Feature

In [46]:
X_train_clustered = X_train.copy()
X_test_clustered = X_test.copy()

X_train_clustered["location_cluster"] = train_clusters
X_test_clustered["location_cluster"] = test_clusters

### One_Hot Encoding Clusters

In [47]:
X_train_clustered = pd.get_dummies(
    X_train_clustered,
    columns=["location_cluster"],
    drop_first=True
)

X_test_clustered = pd.get_dummies(
    X_test_clustered,
    columns=["location_cluster"],
    drop_first=True
)

# Align columns
X_train_clustered, X_test_clustered = X_train_clustered.align(
    X_test_clustered,
    join="left",
    axis=1,
    fill_value=0
)

### Cross-Validate (RF)

In [48]:
rf_grid_clustered = rf_grid.fit(X_train_clustered, y_train)

In [49]:
best_rf_clustered = rf_grid_clustered.best_estimator_
rf_scores_clustered = cross_validate_model(best_rf_clustered, X_train_clustered, y_train, kf)

print("RF CV RMSE (with clusters):", rf_scores_clustered.mean())
print("RF CV Std:", rf_scores_clustered.std())


RF CV RMSE (with clusters): 0.6348203061072917
RF CV Std: 0.0305473320699378


### Test Evaluation

In [50]:
best_rf_clustered.fit(X_train_clustered, y_train)

preds = best_rf_clustered.predict(X_test_clustered)
test_rmse = np.sqrt(mean_squared_error(y_test, preds))

print("RF Test RMSE (with clusters):", test_rmse)


RF Test RMSE (with clusters): 0.6640046284104295


In [51]:
best_rf.fit(X_train, y_train)

preds_no_cluster = best_rf.predict(X_test)
test_rmse_no_cluster = np.sqrt(mean_squared_error(y_test, preds_no_cluster))

print("RF Test RMSE (no clusters):", test_rmse_no_cluster)

RF Test RMSE (no clusters): 0.6670163170173896


### evaluate different k values

In [52]:
for k in [6, 8, 10, 12, 15]:
    
    kmeans = KMeans(n_clusters=k, algorithm='lloyd', init='k-means++', max_iter=100, n_init=10, tol=0.0001, random_state=42)
    
    train_clusters = kmeans.fit_predict(coords_train_scaled)
    test_clusters = kmeans.predict(coords_test_scaled)
    
    X_tr = X_train.copy()
    X_te = X_test.copy()
    
    X_tr["location_cluster"] = train_clusters
    X_te["location_cluster"] = test_clusters
    
    X_tr = pd.get_dummies(X_tr, columns=["location_cluster"], drop_first=True)
    X_te = pd.get_dummies(X_te, columns=["location_cluster"], drop_first=True)
    
    X_tr, X_te = X_tr.align(X_te, join="left", axis=1, fill_value=0)
    
    best_rf_clustered.fit(X_tr, y_train)
    preds = best_rf_clustered.predict(X_te)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    print(f"k={k}, Test RMSE:", rmse)


k=6, Test RMSE: 0.6641188529514471
k=8, Test RMSE: 0.6632614003611419
k=10, Test RMSE: 0.6640046284104295
k=12, Test RMSE: 0.6638495854228555
k=15, Test RMSE: 0.6673399253899744


### Cross-Validate (Ensamble)

In [53]:
final_scores = cross_validate_model(rf_xgb_model, X_train_clustered, y_train, kf)
print("Ensamble CV RMSE (with clusters):", -final_scores.mean())
print("Ensamble CV Std:", final_scores.std())

Ensamble CV RMSE (with clusters): -0.6336530134499562
Ensamble CV Std: 0.0312819676730555


### Test Evaluation

In [54]:
rf_xgb_model.fit(X_train_clustered, y_train)

preds = rf_xgb_model.predict(X_test_clustered)
test_rmse = np.sqrt(mean_squared_error(y_test, preds))

print("Ensamble Test RMSE (with clusters):", test_rmse)


Ensamble Test RMSE (with clusters): 0.6657055471204864


In [55]:
rf_xgb_model.fit(X_train, y_train)

preds_no_cluster = rf_xgb_model.predict(X_test)
test_rmse_no_cluster = np.sqrt(mean_squared_error(y_test, preds_no_cluster))

print("Ensamble Test RMSE (no clusters):", test_rmse_no_cluster)

Ensamble Test RMSE (no clusters): 0.6645405109211594


In [56]:
X_train_only_cluster = X_train_clustered.drop(columns=["latitude", "longitude"])
X_test_only_cluster = X_test_clustered.drop(columns=["latitude", "longitude"])
rf_xgb_model.fit(X_train_only_cluster, y_train)

preds_no_cluster = rf_xgb_model.predict(X_test_only_cluster)
test_rmse_no_cluster = np.sqrt(mean_squared_error(y_test, preds_no_cluster))

print("Ensamble Test RMSE (only clusters):", test_rmse_no_cluster)

Ensamble Test RMSE (only clusters): 0.6902529344770303


# Interaction Features ((NO significant improvment noticed))

In [57]:
X_test_interaction_features = X_test.copy()
X_test_interaction_features['sft/bedrooms'] = X_test_interaction_features['surface_area'] / X_test_interaction_features['bedrooms'].replace(0, 0.1)
X_test_interaction_features['sft/bathrooms'] = X_test_interaction_features['surface_area'] / X_test_interaction_features['bathrooms'].replace(0, 0.1)
X_test_interaction_features['bed/bath'] = X_test_interaction_features['bedrooms'] / X_test_interaction_features['bathrooms'].replace(0, 0.1) + 1

X_train_interaction_features = X_train.copy()
X_train_interaction_features['sft/bedrooms'] = X_train_interaction_features['surface_area'] / (X_train_interaction_features['bedrooms'] + 1)
X_train_interaction_features['sft/bathrooms'] = X_train_interaction_features['surface_area'] / (X_train_interaction_features['bathrooms'] + 1)
X_train_interaction_features['bed/bath'] = X_train_interaction_features['bedrooms'] / (X_train_interaction_features['bathrooms'] + 1)
X_test_interaction_features.head()

,bedrooms,bathrooms,surface_area,latitude,longitude,furnished?,property_mortgaged?,city_Al Khoms,city_Benghazi,city_Misratah,...,facade_Northern,facade_Northwest,facade_Southeast,facade_Southern,facade_Southwest,facade_Unknown,facade_Western,sft/bedrooms,sft/bathrooms,bed/bath
752,7.0,4.0,6.552508,32.066647,20.112839,False,False,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.936073,1.638127,2.75
765,7.0,4.0,6.216606,32.834011,13.343105,False,False,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.888087,1.554152,2.75
1655,3.0,2.0,5.398163,32.032532,20.062571,False,False,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.799388,2.699081,2.50
1252,3.0,2.0,6.293419,32.091995,20.091747,False,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.097806,3.146710,2.50
1102,5.0,2.0,5.379897,32.838144,13.360048,False,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.075979,2.689949,3.50


In [58]:
top_features = ['bedrooms', 'bathrooms', 'surface_area', 'latitude', 'longitude']

X_train_top_features = X_train[top_features]
X_test_top_features = X_test[top_features]

In [59]:
X_train_top_features.head()

,bedrooms,bathrooms,surface_area,latitude,longitude
570,4.0,2.0,5.111988,31.921021,19.963049
841,7.0,4.0,5.525453,32.133366,20.103369
1674,4.0,4.0,5.707110,32.032532,20.062571
915,3.0,3.0,5.117994,32.895565,13.211289
864,3.0,2.0,5.111988,32.869293,13.236674


In [60]:
rf_grid_top_features = rf_grid.fit(X_train_top_features, y_train)

In [61]:
best_rf_top_features = rf_grid_top_features.best_estimator_
rf_scores_top_features = cross_validate_model(best_rf_top_features, X_train_top_features, y_train, kf)

print(rf_scores_top_features)
print("RF CV RMSE (top features):", rf_scores_top_features.mean())
print("RF CV Std:", rf_scores_top_features.std())

[0.63113544 0.67810035 0.70902032 0.65202968 0.63721906 0.62857589
 0.6491869  0.70200019 0.59415711 0.62898848]
RF CV RMSE (top features): 0.6510413422916213
RF CV Std: 0.03392508395657297


# removing outliers (NO significant improvment noticed)

In [62]:
filtered_df = df.copy()

In [63]:
filtered_df['price'] = np.exp(filtered_df['price'])

In [64]:
filtered_df['price'].describe()

count    1.971000e+03
mean     7.485413e+05
std      7.655898e+05
min      8.500100e+04
25%      3.000010e+05
50%      5.000010e+05
75%      9.000010e+05
max      8.000001e+06
Name: price, dtype: float64

In [65]:
filtered_df = filtered_df[filtered_df['price'] < 5000000]

In [66]:
filtered_df['price'].describe()

count    1.961000e+03
mean     7.216598e+05
std      6.644146e+05
min      8.500100e+04
25%      3.000010e+05
50%      5.000010e+05
75%      8.900010e+05
max      4.585759e+06
Name: price, dtype: float64

In [67]:
filtered_df['price'] = np.log(filtered_df['price'])


In [68]:
X_filtered = filtered_df.drop("price", axis=1)
y_filtered = filtered_df["price"]

In [69]:
X_train_filtered, X_test_filtered, y_train_filtered, y_test_filtered = train_test_split(
    X_filtered,
    y_filtered,
    test_size=0.2,
    random_state=rs
)

In [70]:
rf_grid_filtered = rf_grid.fit(X_train_filtered, y_train_filtered)

In [71]:
best_rf_filtered = rf_grid_filtered.best_estimator_
rf_scores_filtered = cross_validate_model(best_rf_filtered, X_train_filtered, y_train_filtered, kf)

print("RF CV RMSE (filtered):", rf_scores_filtered.mean())
print("RF CV Std:", rf_scores_filtered.std())

print("RF CV RMSE (original):", rf_scores.mean())

RF CV RMSE (filtered): 0.6404554405794466
RF CV Std: 0.027740934456378208
RF CV RMSE (original): 0.6345357902979594


# Conclusion

In [72]:
from sklearn.metrics import mean_absolute_error, r2_score

In [73]:
# Predict log price
y_pred_log = best_rf_top_features.predict(X_test_top_features)

In [74]:
# Metrics in log space
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

print("LOG SPACE METRICS")
print("MAE (log):", mae_log)
print("RMSE (log):", rmse_log)
print("R^2 (log):", r2_log)

LOG SPACE METRICS
MAE (log): 0.5366152699188899
RMSE (log): 0.6713341254714346
R^2 (log): 0.2604048967505941


In [75]:
# Convert back to original price scale
y_pred = np.exp(y_pred_log)
y_true = np.exp(y_test)

In [76]:
# Metrics in original price space
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("\nORIGINAL PRICE SPACE METRICS")
print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)


ORIGINAL PRICE SPACE METRICS
MAE: 395632.9546167123
RMSE: 669083.0089255567
R^2: 0.17597686859777906


In [77]:
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
print("MAPE (%):", mape)

MAPE (%): 61.588311239860985


In [78]:
df_real_price = np.exp(df['price'])

df_real_price.describe()


count    1.971000e+03
mean     7.485413e+05
std      7.655898e+05
min      8.500100e+04
25%      3.000010e+05
50%      5.000010e+05
75%      9.000010e+05
max      8.000001e+06
Name: price, dtype: float64

# Model

In [79]:
def predict_price(bedrooms, bathrooms, surface_area, latitude, longitude):
    suf = np.log(surface_area)
    x = pd.DataFrame(
        [[bedrooms, bathrooms, suf, latitude, longitude]],
        columns=X_train_top_features.columns
    )

    # Make prediction using the trained model
    log_pred = best_rf_top_features.predict(x)
    
    # Return the predicted price (exponentiate log prediction)
    return np.exp(log_pred[0])

In [80]:
predict_price(3, 1, 70, 50.7128, -74.0060)

np.float64(592048.9058882314)

In [81]:
import pickle

In [82]:
with open("libya_house_price_model.pkl", "wb") as f:
    pickle.dump(best_rf_top_features, f)